# 🚀 MINav Offline RL Training on Kaggle (Real-World Dataset)
### End-to-End Pipeline: Real-World Dataset → DINOv3 (ViT-S) Embeddings → Hindsight Relabeling → TD3+BC Offline RL

This notebook processes your uploaded real-world Ranger Mini V3 exploration dataset (`run_full_main`, ~32.6k frames, 33 segments) and trains an offline goal-conditioned navigation policy with TD3+BC.

#### ⚙️ Kaggle Recommended Settings:
- **Accelerator**: **GPU T4 x2** or **GPU P100**
- **Internet**: **ON** (required for `git clone` & downloading DINOv3 pretrained weights)
- **Persistence**: Files in `/kaggle/working/` persist during session


In [ ]:
# Cell 1: Environment & GPU Verification
import os, sys, subprocess, torch

print('=' * 60)
print('SYSTEM & GPU STATUS')
print('=' * 60)
print(f'CUDA Available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name    : {torch.cuda.get_device_name(0)}')
    print(f'Device Count   : {torch.cuda.device_count()}')
    print(f'VRAM Total     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ WARNING: GPU not detected! Make sure to select GPU T4 x2 or P100 in Kaggle Settings.')
print('=' * 60)

# Install required dependencies
!pip install -q pyarrow pandas pyyaml wandb tensorboard timm torchvision
print('✓ Dependencies successfully installed.')


In [ ]:
# Cell 2: Clone / Update Repository
REPO_DIR = '/kaggle/working/RL-Image-Goal-Navigation'

if not os.path.exists(REPO_DIR):
    print(f'Cloning repository into {REPO_DIR}...')
    !git clone https://github.com/sanmita13742/RL-Image-Goal-Navigation.git {REPO_DIR}
else:
    print('Repository exists. Fetching latest updates from origin/main...')
    !git -C {REPO_DIR} reset --hard origin/main
    !git -C {REPO_DIR} pull origin main

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'✓ Active Working Directory: {os.getcwd()}')


In [ ]:
# Cell 3: Auto-Detect Uploaded Kaggle Dataset
from pathlib import Path
import shutil

print('=' * 60)
print('SEARCHING FOR EXPLORATION DATASET IN /kaggle/input')
print('=' * 60)

input_dir = Path('/kaggle/input')
explore_dir = None

# Scan for exploration_metadata.json or segment_000
for p in input_dir.rglob('exploration_metadata.json'):
    explore_dir = p.parent
    print(f'✓ Found exploration metadata at: {p}')
    break

if explore_dir is None:
    for p in input_dir.rglob('segment_000'):
        explore_dir = p.parent
        print(f'✓ Found segment_000 at: {p}')
        break

if explore_dir is None:
    print('❌ Exploration dataset not found in /kaggle/input.')
    print('Folders found in /kaggle/input:')
    for item in input_dir.iterdir():
        print(' -', item)
    raise FileNotFoundError('Please attach your uploaded dataset to the Kaggle notebook.')

print(f'\nTarget Exploration Directory: {explore_dir}')

# Set up execution run directory
RUN_NAME = 'realworld_run'
RUN_DIR = Path(f'/kaggle/working/runs/{RUN_NAME}')
RUN_DIR.mkdir(parents=True, exist_ok=True)
EXPLORATION_LINK = RUN_DIR / 'exploration'

# Create symlink to save disk and allow zero-copy loading
if EXPLORATION_LINK.exists():
    if EXPLORATION_LINK.is_symlink():
        EXPLORATION_LINK.unlink()
    else:
        shutil.rmtree(EXPLORATION_LINK)

os.symlink(str(explore_dir), str(EXPLORATION_LINK))
print(f'✓ Symlinked: {explore_dir} -> {EXPLORATION_LINK}')

segments = sorted(EXPLORATION_LINK.glob('segment_*'))
print(f'✓ Total segments detected: {len(segments)}')


In [ ]:
# Cell 4: Configure Run Parameters
import yaml

config_file = Path(REPO_DIR) / 'realworld' / 'configs' / 'realworld_pipeline.yaml'
with open(config_file, 'r') as f:
    config = yaml.safe_load(f)

# User Hyperparameters
TRAIN_STEPS = 200_000       # Total gradient steps for TD3+BC (e.g. 100k - 500k)
BATCH_SIZE = 256            # Standard MINav mini-batch size
CHECKPOINT_EVERY = 25_000   # Checkpoint saving interval
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

config['training']['total_gradient_steps'] = TRAIN_STEPS
config['training']['batch_size'] = BATCH_SIZE
config['training']['checkpoint_every'] = CHECKPOINT_EVERY

CONFIG_RUN = RUN_DIR / 'config_run.yaml'
with open(CONFIG_RUN, 'w') as f:
    yaml.dump(config, f)

print(f'✓ Run configuration written to: {CONFIG_RUN}')
print(f'  - Accelerator: {DEVICE}')
print(f'  - Target Steps: {TRAIN_STEPS:,}')
print(f'  - Batch Size: {BATCH_SIZE}')


In [ ]:
# Cell 5: Phase 2 — DINOv3 Encoding & Hindsight Dataset Generation
print('=' * 60)
print('PHASE 2: DINOv3 (ViT-Small) ENCODING & HINDSIGHT GOALS')
print('=' * 60)

cmd = [
    sys.executable, str(Path(REPO_DIR) / 'realworld' / 'scripts' / 'build_dataset.py'),
    '--config', str(CONFIG_RUN),
    '--run-dir', str(RUN_DIR),
    '--device', DEVICE
]

print('Executing command:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('\n✓ Phase 2 completed successfully!')


In [ ]:
# Cell 6: Verify Processed Embeddings & Transitions
import numpy as np
import pandas as pd

processed_dir = RUN_DIR / 'processed'
dinov3_dir = processed_dir / 'dinov3'
hindsight_dir = processed_dir / 'hindsight'

print('=' * 60)
print('DATASET AUDIT & SHAPE VERIFICATION')
print('=' * 60)

phi_path = dinov3_dir / 'phi_vectors.npy'
if phi_path.exists():
    phi = np.load(phi_path, mmap_mode='r')
    print(f'✓ DINOv3 Embeddings Shape : {phi.shape} (Expected: [N, 384])')
    print(f'  L2 Unit Norm Check      : {np.linalg.norm(phi[0]):.4f} (Expected: 1.0000)')

ssd_path = dinov3_dir / 'ssd_scores.npy'
if ssd_path.exists():
    ssd = np.load(ssd_path)
    print(f'✓ SSD Scores Count        : {len(ssd)} (Mean: {ssd.mean():.4f})')

geom_path = hindsight_dir / 'geometric_transitions.parquet'
if geom_path.exists():
    df_geom = pd.read_parquet(geom_path)
    print(f'✓ Geometric Transitions   : {len(df_geom):,} transitions')

unif_path = hindsight_dir / 'uniform_transitions.parquet'
if unif_path.exists():
    df_unif = pd.read_parquet(unif_path)
    print(f'✓ Uniform Transitions     : {len(df_unif):,} transitions')
print('=' * 60)


In [ ]:
# Cell 7: Phase 3 — Train Offline TD3+BC Agent
print('=' * 60)
print('PHASE 3: OFFLINE TD3+BC ACTOR-CRITIC TRAINING')
print('=' * 60)

cmd = [
    sys.executable, str(Path(REPO_DIR) / 'realworld' / 'scripts' / 'train.py'),
    '--config', str(CONFIG_RUN),
    '--run-dir', str(RUN_DIR),
    '--device', DEVICE
]

print('Executing command:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('\n✓ Phase 3 TD3+BC training complete!')


In [ ]:
# Cell 8: Checkpoint Audit & Summary
train_dir = RUN_DIR / 'training'
checkpoints = sorted(train_dir.glob('*.pt'))

print('=' * 60)
print('TRAINED MODEL CHECKPOINTS')
print('=' * 60)
for ckpt in checkpoints:
    print(f'✓ {ckpt.name} ({ckpt.stat().st_size / 1e6:.2f} MB)')
print('=' * 60)


In [ ]:
# Cell 9: Package and Export Trained Models for Download
import zipfile

OUTPUT_ZIP = '/kaggle/working/minav_realworld_policy.zip'
print(f'Creating downloadable archive at {OUTPUT_ZIP}...')

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for f in train_dir.rglob('*'):
        if f.is_file():
            zipf.write(f, arcname=f'training/{f.name}')
    if (RUN_DIR / 'config_run.yaml').exists():
        zipf.write(RUN_DIR / 'config_run.yaml', arcname='config_run.yaml')

print(f'✓ Archive generated! Size: {os.path.getsize(OUTPUT_ZIP) / 1e6:.2f} MB')
print('Download "minav_realworld_policy.zip" directly from the Kaggle Output panel.')
